In [2]:
!pip install mediapipe -q
!pip install scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 15.4 MB/s eta 0:00:00


In [3]:
import os
import cv2
import numpy as np
import mediapipe as mp
import zipfile
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import pickle
from google.colab import files

In [4]:
!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

print("Model file downloaded!")

Model file downloaded!


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
zip_name = '/content/drive/MyDrive/SCT_TASK_4_Hand Gesture Recognition System.zip'

import zipfile
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/dataset')

print("Extracted successfully!")

Extracted successfully!


In [7]:
# New API setup for version 0.10+
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

# Create the landmarker with IMAGE mode (since we process images not video)
options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
    running_mode=VisionRunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.3,
    min_hand_presence_confidence=0.3,
    min_tracking_confidence=0.3
)

landmarker = HandLandmarker.create_from_options(options)

print("MediaPipe Hand Landmarker ready!")

MediaPipe Hand Landmarker ready!


In [10]:
import os

for root, dirs, files in os.walk('/content/dataset'):
    level = root.replace('/content/dataset', '').count(os.sep)
    if level > 4:
        break
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")

dataset/
  SCT_TASK_4_Hand Gesture Recognition System/
    leapGestRecog/
      01/
        01_palm/
        06_index/
        04_fist_moved/
        05_thumb/
        02_l/
        03_fist/
        07_ok/
      00/
        01_palm/
        10_down/
        09_c/
        06_index/
        04_fist_moved/
        05_thumb/
        02_l/
        03_fist/
        08_palm_moved/
        07_ok/


In [11]:
data = []
labels = []
skipped = 0

dataset_path = '/content/dataset/SCT_TASK_4_Hand Gesture Recognition System/leapGestRecog'

print("Starting landmark extraction...\n")

for subject in sorted(os.listdir(dataset_path)):
    subject_path = os.path.join(dataset_path, subject)

    if not os.path.isdir(subject_path):
        continue

    print(f"Processing subject: {subject}")

    for gesture_folder in sorted(os.listdir(subject_path)):
        gesture_path = os.path.join(subject_path, gesture_folder)

        if not os.path.isdir(gesture_path):
            continue

        gesture_name = gesture_folder

        for img_file in os.listdir(gesture_path):
            img_path = os.path.join(gesture_path, img_file)

            img = cv2.imread(img_path)
            if img is None:
                continue

            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=img_rgb
            )

            result = landmarker.detect(mp_image)

            if result.hand_landmarks:
                hand = result.hand_landmarks[0]

                landmark_values = []
                for lm in hand:
                    landmark_values.append(lm.x)
                    landmark_values.append(lm.y)
                    landmark_values.append(lm.z)

                data.append(landmark_values)
                labels.append(gesture_name)
            else:
                skipped += 1

print(f"\nDone!")
print(f"Images with hand detected : {len(data)}")
print(f"Images skipped (no hand)  : {skipped}")
print(f"Gesture classes found     : {sorted(set(labels))}")

Starting landmark extraction...

Processing subject: 00
Processing subject: 01

Done!
Images with hand detected : 1183
Images skipped (no hand)  : 2040
Gesture classes found     : ['01_palm', '02_l', '03_fist', '06_index', '07_ok', '09_c', '10_down']


In [14]:
# Convert to numpy arrays
X = np.array(data)
y = np.array(labels)

# Check how many samples per gesture
print("Samples per gesture:")
unique, counts = np.unique(y, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {u} : {c} images")

# Remove gestures with less than 5 samples
valid_idx = []
for i, label in enumerate(y):
    count = list(counts)[list(unique).index(label)]
    if count >= 5:
        valid_idx.append(i)

X = X[valid_idx]
y = y[valid_idx]

print(f"\nSamples after filtering: {len(X)}")

# Convert gesture names to numbers
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("\nFinal gesture classes:")
for i, name in enumerate(le.classes_):
    print(f"  {i} → {name}")

# Split without stratify to avoid the error
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42
)

print(f"\nTraining samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")

# Train Random Forest
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("\nModel trained successfully!")

Samples per gesture:
  01_palm : 391 images
  02_l : 220 images
  03_fist : 11 images
  06_index : 191 images
  07_ok : 223 images
  09_c : 1 images
  10_down : 146 images

Samples after filtering: 1182

Final gesture classes:
  0 → 01_palm
  1 → 02_l
  2 → 03_fist
  3 → 06_index
  4 → 07_ok
  5 → 10_down

Training samples : 945
Testing samples  : 237

Model trained successfully!


In [15]:
def predict_gesture(image_path):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    result = landmarker.detect(mp_image)

    if result.hand_landmarks:
        hand = result.hand_landmarks[0]

        # Extract landmarks
        landmark_values = []
        for lm in hand:
            landmark_values.append(lm.x)
            landmark_values.append(lm.y)
            landmark_values.append(lm.z)

        # Predict gesture
        prediction = model.predict([landmark_values])[0]
        gesture_name = le.inverse_transform([prediction])[0]
        confidence = max(model.predict_proba([landmark_values])[0]) * 100

        # Draw landmarks on image manually
        h, w, _ = img.shape
        for lm in hand:
            cx, cy = int(lm.x * w), int(lm.y * h)
            cv2.circle(img, (cx, cy), 5, (0, 255, 0), -1)

        # Show result
        plt.figure(figsize=(6, 5))
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.title(f"Gesture: {gesture_name}  |  Confidence: {confidence:.1f}%", fontsize=13)
        plt.axis('off')
        plt.show()

        print(f"Predicted Gesture : {gesture_name}")
        print(f"Confidence        : {confidence:.1f}%")

    else:
        print("No hand detected in this image")

# Change this path to any image from your dataset
predict_gesture('/content/dataset/SCT_TASK_4_Hand Gesture Recognition System/leapGestRecog/00/01_palm/frame_00_01_0001.png')

No hand detected in this image


In [17]:
with open('gesture_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print("Saved!")


Saved!


In [18]:
from google.colab import files

files.download('gesture_model.pkl')
files.download('label_encoder.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>